# TopN Rolling Backtest Research Notebook

Validates the **winner-bucket strategy** by calling the same pure functions
used by `run_backtest()` in `src/backtest_strategy.py`.

No separate engine — same code, interactive access.

## Pipeline
```
qlib data  →  winner_prob (LGBMClassifier)
           →  select_topn_with_guardrail()   ← same fn as run_backtest()
           →  compute_portfolio_returns()    ← same fn as run_backtest()
           →  build_backtest_summary()       ← same fn as run_backtest()
```

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import qlib
from qlib.data import D
from qlib.contrib.data.handler import Alpha158

from src.research.backtest_core import (
    select_topn_with_guardrail,
    select_topn_no_guardrail,
    compute_portfolio_returns,
    compute_turnover,
    build_backtest_summary,
)
from src.common.config_utils import load_watchlist
from src.common.paths import MODELS_DIR, PROJECT_ROOT

## 1. Config — edit these to match your run

In [ ]:
START_DATE  = '2025-10-23'
END_DATE    = '2026-01-23'
BENCHMARK   = 'QQQ'
TOPK_LIST   = [5, 10, 20]   # evaluate multiple basket sizes
INITIAL_CASH = 10_000
MODEL_PATH  = MODELS_DIR / 'us_classifier.pkl'
DATA_DIR    = 'data/watchlist'
WATCHLIST   = 'configs/watchlist.yaml'

## 2. Initialise qlib + load model

In [ ]:
from src.common.qlib_utils import ensure_qlib_init
ensure_qlib_init(provider_uri=DATA_DIR, region='us')

with open(MODEL_PATH, 'rb') as f:
    model = pickle.load(f)
print(type(model))

## 3. Load features + predict winner_prob

In [ ]:
watchlist   = load_watchlist(WATCHLIST)
us_tickers  = watchlist.get('us', [])
if BENCHMARK not in us_tickers:
    us_tickers.append(BENCHMARK)

handler_kwargs = {
    'start_time': START_DATE, 'end_time': END_DATE,
    'fit_start_time': START_DATE, 'fit_end_time': START_DATE,
    'instruments': us_tickers,
    'infer_processors': [
        {'class': 'RobustZScoreNorm', 'kwargs': {'fields_group': 'feature', 'clip_outlier': True}},
        {'class': 'Fillna', 'kwargs': {'fields_group': 'feature'}},
    ],
    'learn_processors': [{'class': 'DropnaLabel'}],
    'label': ['Ref($close, -20) / $close - 1'],
}
dh = Alpha158(**handler_kwargs)
df_data = dh.fetch(selector=(START_DATE, END_DATE), col_set='feature')

# LGBMClassifier: predict_proba returns [prob_0, prob_1]
winner_prob = pd.Series(model.predict_proba(df_data.values)[:, 1], index=df_data.index)
winner_prob.head()

## 4. Load market data (prices + MA60)

In [ ]:
fields = ['$close', 'Mean($close, 60)']
names  = ['close', 'ma60']
market_data = D.features(us_tickers, fields, start_time=START_DATE, end_time=END_DATE)
market_data.columns = names

close_prices = market_data['close'].unstack(level='instrument')
ma60_data    = market_data['ma60'].unstack(level='instrument')
daily_rets   = close_prices.pct_change()
bench_ret    = daily_rets[BENCHMARK] if BENCHMARK in daily_rets.columns else pd.Series(0, index=daily_rets.index)
print('Universe:', close_prices.shape[1], 'tickers | Date range:', close_prices.index[0], '->', close_prices.index[-1])

## 5. Build holdings by date for each TopK

In [ ]:
trading_days = sorted(daily_rets.index)

def build_holdings(topk, use_guardrail=True):
    holdings = {}
    for date in trading_days:
        try:
            scores = winner_prob.xs(date, level='datetime')
        except KeyError:
            holdings[date] = []
            continue
        if use_guardrail:
            holdings[date] = select_topn_with_guardrail(
                scores, topk=topk,
                prices=close_prices.loc[date],
                ma60=ma60_data.loc[date],
            )
        else:
            holdings[date] = select_topn_no_guardrail(scores, topk=topk)
    return holdings

all_holdings = {k: build_holdings(k) for k in TOPK_LIST}
print('Holdings built for TopK:', list(all_holdings.keys()))

## 6. Simulate portfolio returns + summary table

In [ ]:
results = {}
portfolios = {}
for k in TOPK_LIST:
    hist_df  = compute_portfolio_returns(all_holdings[k], daily_rets, bench_ret, INITIAL_CASH)
    turnover = compute_turnover(all_holdings[k])
    summary  = build_backtest_summary(hist_df, bench_ret, INITIAL_CASH, turnover)
    portfolios[k] = hist_df
    results[k] = summary.__dict__

pd.DataFrame(results).T

## 7. Equity curve vs benchmark

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
for k, pf in portfolios.items():
    (pf['portfolio_value'] / INITIAL_CASH).plot(ax=ax, label=f'Top{k}')
(portfolios[TOPK_LIST[0]]['benchmark_value'] / INITIAL_CASH).plot(ax=ax, label=BENCHMARK, linestyle='--', color='grey')
ax.set_title('Equity Curve vs Benchmark')
ax.set_ylabel('Normalised value (1 = start)')
ax.legend(); plt.tight_layout(); plt.show()

## 8. Cumulative excess alpha

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
for k, pf in portfolios.items():
    pf['excess_alpha'].fillna(0).cumsum().plot(ax=ax, label=f'Top{k}')
ax.axhline(0, color='grey', linewidth=0.8)
ax.set_title('Cumulative Excess Alpha (daily, vs benchmark)')
ax.legend(); plt.tight_layout(); plt.show()

## 9. TopN vs BottomN spread (no guardrail)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
for k in TOPK_LIST:
    bottom_holdings = build_holdings(k, use_guardrail=False)
    bottom_hist = compute_portfolio_returns(bottom_holdings, daily_rets, bench_ret, INITIAL_CASH)
    spread = portfolios[k]['daily_return'] - bottom_hist['daily_return']
    spread.fillna(0).cumsum().plot(ax=ax, label=f'Top{k}-Bottom{k}')
ax.axhline(0, color='grey', linewidth=0.8)
ax.set_title('Cumulative Top-Bottom Spread')
ax.legend(); plt.tight_layout(); plt.show()

## 10. Turnover inspection

In [ ]:
turn_df = pd.DataFrame({f'Top{k}': compute_turnover(all_holdings[k]) for k in TOPK_LIST})
turn_df.describe().T